# Diagnóstico aprofundado — PyArrow → pandas

## Objetivo

Investigar por que a leitura das bases `grupo` e `dsp` aumenta o USS muito mais do que a memória profunda dos DataFrames resultantes.

O notebook separa quatro etapas:

1. Leitura dos dados Parquet para uma `pyarrow.Table`.
2. Conversão explícita da tabela Arrow para pandas.
3. Conversão de colunas textuais repetitivas para `category`.
4. Comparação com `pd.read_parquet()` e com leitura Arrow usando codificação dictionary.

Os testes são executados em processos filhos independentes. Isso é importante porque os testes anteriores mostraram que o kernel do notebook retém memória após algumas leituras.

**Não altera arquivos, Parquets ou funções do dashboard.** O foco inicial são `grupo` e `dsp`, com possibilidade de incluir as outras bases.

## 1. Configuração

Execute a partir da raiz de `app_render`. A configuração é importada de `dash_utils.config`, e as projeções são as mesmas do `features_names.py` utilizado no dashboard.

In [1]:
from pathlib import Path
import os, sys, json, subprocess, textwrap, importlib
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dash_utils" / "config.py").exists():
    candidatos = [
        Path.cwd().parent,
        Path.cwd() / "app_render",
        Path.home() / "PycharmProjects" / "Projeto_ENEM" / "app_render",
    ]
    PROJECT_ROOT = next(
        (p.resolve() for p in candidatos if (p / "dash_utils" / "config.py").exists()),
        None
    )
if PROJECT_ROOT is None:
    raise FileNotFoundError("Não encontrei dash_utils/config.py. Ajuste PROJECT_ROOT.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

config = importlib.import_module("dash_utils.config")
CONSUMO_DIR = Path(config.CONSUMO_DIR)
UF = "PA"
REFERENCIA = "ESC"   # "ESC", "PROVA" ou None

UF_DIR = CONSUMO_DIR / f"uf={UF}"
ARQUIVOS = {
    "grupo": UF_DIR / "desempenho_grupos_municipal.parquet",
    "dsp": UF_DIR / "desempenho_sexo_faixa_etaria_municipal.parquet",
    "painel": UF_DIR / "painel_municipal.parquet",
    "faixa": UF_DIR / "faixa_etaria_municipal.parquet",
}

COLUNAS = {
    "grupo": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "tipo_grupo", "grupo", "disciplina", "media", "n_validos",
        "mediana", "std", "q1", "q3",
    ],
    "dsp": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "sexo", "faixa_etaria", "disciplina", "n_participantes",
        "n_validos", "media", "std",
    ],
    "painel": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "n_participantes", "n_masculino", "n_feminino",
        "n_faixa_etaria_informada", "n_escola_federal",
        "n_escola_estadual", "n_escola_municipal", "n_escola_privada",
        "n_escola_nao_informada", "media_redacao", "n_validos_redacao",
        "media_objetiva", "n_validos_objetiva", "media_cn", "n_validos_cn",
        "media_ch", "n_validos_ch", "media_lc", "n_validos_lc",
        "media_mt", "n_validos_mt",
    ],
    "faixa": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL", "TP_FAIXA_ETARIA", "n_participantes",
    ],
}

def filtros_ref(ref):
    if ref == "ESC":
        vals = ["ESC", "RESIDENCIA"]
    elif ref == "PROVA":
        vals = ["PROVA"]
    else:
        return None
    return [("TIPO_REFERENCIA_TERRITORIAL", "in", vals)]

print("Projeto:", PROJECT_ROOT)
print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("UF:", UF, "| referência:", REFERENCIA)
for nome, arq in ARQUIVOS.items():
    print(nome, arq.exists(), arq.stat().st_size / 1024**2 if arq.exists() else None)

Projeto: /home/akel/PycharmProjects/Projeto_ENEM/app_render
Python: 3.11.18 (main, Oct 26 2026, 11:35:17) [GCC 15.3.0]
Pandas: 3.1.0
PyArrow: 26.0.1
UF: PA | referência: ESC
grupo True 8.573951721191406
dsp True 10.102821350097656
painel True 1.1853256225585938
faixa True 0.14458465576171875


## 2. Inspeção dos metadados físicos

Como o primeiro diagnóstico encontrou apenas um Row Group em cada arquivo, esta etapa examina o tamanho comprimido/descomprimido e os encodings de cada coluna projetada. Isso ajuda a avaliar se a leitura está dominada por descompressão, strings ou conversão de tipos.

In [2]:
def metadados_colunas(base):
    pf = pq.ParquetFile(ARQUIVOS[base])
    meta = pf.metadata
    rows = []
    for i in range(meta.num_row_groups):
        rg = meta.row_group(i)
        for j in range(rg.num_columns):
            c = rg.column(j)
            if c.path_in_schema in COLUNAS[base]:
                rows.append({
                    "base": base,
                    "row_group": i,
                    "coluna": c.path_in_schema,
                    "codec": c.compression,
                    "encodings": ", ".join(map(str, c.encodings)),
                    "comprimido_mb": c.total_compressed_size / 1024**2,
                    "descomprimido_mb": c.total_uncompressed_size / 1024**2,
                })
    return pd.DataFrame(rows)

metadados = pd.concat(
    [metadados_colunas(b) for b in ["grupo", "dsp"]],
    ignore_index=True
)
display(metadados.sort_values(["base", "descomprimido_mb"], ascending=[True, False]))

,base,row_group,coluna,codec,encodings,comprimido_mb,descomprimido_mb
21,dsp,0,std,ZSTD,"PLAIN, RLE, RLE_DICTIONARY",2.560422,2.728272
20,dsp,0,media,ZSTD,"PLAIN, RLE, RLE_DICTIONARY",1.972273,2.621185
19,dsp,0,n_validos,ZSTD,"PLAIN, RLE, RLE_DICTIONARY",0.389273,0.618356
18,dsp,0,n_participantes,ZSTD,"PLAIN, RLE, RLE_DICTIONARY",0.392150,0.572165
16,dsp,0,faixa_etaria,ZSTD,"PLAIN, RLE, RLE_DICTIONARY",0.256254,0.270662
17,dsp,0,disciplina,ZSTD,"PLAIN, RLE, RLE_DICTIONARY",0.117662,0.153714
15,dsp,0,sexo,ZSTD,"PLAIN, RLE, RLE_DICTIONARY",0.056638,0.056410
13,dsp,0,CO_MUNICIPIO_TERRITORIO,ZSTD,"PLAIN, RLE, RLE_DICTIONARY",0.023168,0.031111
12,dsp,0,ano,ZSTD,"PLAIN, RLE, RLE_DICTIONARY",0.002069,0.001966
14,dsp,0,TIPO_REFERENCIA_TERRITORIAL,ZSTD,"PLAIN, RLE, RLE_DICTIONARY",0.001425,0.001219


## 3. Medição por processo isolado

O worker registra RSS/USS em cada fronteira:

- depois dos imports;
- depois da leitura Parquet → Arrow;
- depois da conversão Arrow → pandas;
- depois da conversão de strings selecionadas para `category`;
- depois de liberar os objetos.

Também registra `Table.nbytes`, `Table.get_total_buffer_size()` e `DataFrame.memory_usage(deep=True)`.

**Interpretação:** `Table.nbytes`/buffers Arrow e memória profunda pandas são medidas de representação dos dados; RSS/USS medem a memória do processo. Não somar essas medidas entre si.

In [3]:
WORKER = r'''
import sys, json, gc, time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import psutil

p = json.loads(sys.argv[1])
proc = psutil.Process()

def mem():
    x = proc.memory_full_info()
    return {"rss": x.rss / 1024**2, "uss": getattr(x, "uss", 0) / 1024**2}

def snap(nome, out):
    m = mem()
    out[nome + "_rss_mb"] = m["rss"]
    out[nome + "_uss_mb"] = m["uss"]

out = {"base": p["base"], "modo": p["modo"], "referencia": p["referencia"]}
snap("apos_imports", out)

filters = None
if p["referencia"] == "ESC":
    filters = [("TIPO_REFERENCIA_TERRITORIAL", "in", ["ESC", "RESIDENCIA"])]
elif p["referencia"] == "PROVA":
    filters = [("TIPO_REFERENCIA_TERRITORIAL", "in", ["PROVA"])]

t0 = time.perf_counter()
pf = pq.ParquetFile(p["arquivo"])
table = pf.read_row_group(0, columns=p["columns"])
if filters:
    # Aplica o mesmo critério territorial no Arrow após leitura.
    col = table["TIPO_REFERENCIA_TERRITORIAL"]
    vals = ["ESC", "RESIDENCIA"] if p["referencia"] == "ESC" else ["PROVA"]
    mask = pc.is_in(col, value_set=pa.array(vals, type=col.type))
    table = table.filter(mask)
t1 = time.perf_counter()

out["tempo_arrow_leitura_s"] = t1 - t0
out["arrow_linhas"] = table.num_rows
out["arrow_colunas"] = table.num_columns
out["arrow_nbytes_mb"] = table.nbytes / 1024**2
out["arrow_buffers_mb"] = table.get_total_buffer_size() / 1024**2
snap("apos_arrow", out)

t2 = time.perf_counter()
df = table.to_pandas()
t3 = time.perf_counter()
out["tempo_arrow_para_pandas_s"] = t3 - t2
out["pandas_deep_mb"] = df.memory_usage(index=True, deep=True).sum() / 1024**2
out["pandas_dtypes"] = {str(k): str(v) for k, v in df.dtypes.items()}
snap("apos_pandas", out)

# Converte apenas strings com cardinalidade baixa/moderada.
cols_cat = []
for c in df.select_dtypes(include=["object", "str", "string"]).columns:
    nunique = df[c].nunique(dropna=False)
    ratio = nunique / max(len(df), 1)
    if ratio < p["limiar_cardinalidade"]:
        cols_cat.append(c)

t4 = time.perf_counter()
for c in cols_cat:
    df[c] = df[c].astype("category")
t5 = time.perf_counter()

out["colunas_category"] = cols_cat
out["tempo_category_s"] = t5 - t4
out["pandas_category_deep_mb"] = df.memory_usage(index=True, deep=True).sum() / 1024**2
snap("apos_category", out)

# Liberação explícita e coleta.
del df, table, pf
gc.collect()
snap("apos_del_gc", out)

print(json.dumps(out))
'''

def executar_worker(base, modo="arrow_pandas_category", referencia=REFERENCIA, limiar=0.5):
    payload = {
        "base": base,
        "modo": modo,
        "referencia": referencia,
        "arquivo": str(ARQUIVOS[base]),
        "columns": COLUNAS[base],
        "limiar_cardinalidade": limiar,
    }
    proc = subprocess.run(
        [sys.executable, "-c", WORKER, json.dumps(payload)],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    if proc.returncode:
        raise RuntimeError(
            f"Erro no worker {base}:\\n{proc.stdout}\\n{proc.stderr}"
        )
    linhas = [x for x in proc.stdout.splitlines() if x.strip().startswith("{")]
    if not linhas:
        raise RuntimeError(f"Worker sem JSON: {proc.stdout}")
    return json.loads(linhas[-1])

## 4. Teste principal: Arrow → pandas → category

Executa os testes isolados de `grupo` e `dsp`, usando a referência selecionada no início do notebook.

In [4]:
resultados = []
for base in ["grupo", "dsp"]:
    r = executar_worker(base)
    resultados.append(r)
    print(
        base,
        "| Arrow buffers:", round(r["arrow_buffers_mb"], 1), "MB",
        "| pandas deep:", round(r["pandas_deep_mb"], 1), "MB",
        "| category deep:", round(r["pandas_category_deep_mb"], 1), "MB",
        "| USS após imports:", round(r["apos_imports_uss_mb"], 1), "MB",
        "| USS após Arrow:", round(r["apos_arrow_uss_mb"], 1), "MB",
        "| USS após pandas:", round(r["apos_pandas_uss_mb"], 1), "MB",
        "| USS após category:", round(r["apos_category_uss_mb"], 1), "MB",
    )

df_etapas = pd.DataFrame(resultados)
display(df_etapas[[
    "base", "arrow_linhas", "arrow_nbytes_mb", "arrow_buffers_mb",
    "pandas_deep_mb", "pandas_category_deep_mb",
    "tempo_arrow_leitura_s", "tempo_arrow_para_pandas_s", "tempo_category_s",
    "apos_imports_uss_mb", "apos_arrow_uss_mb", "apos_pandas_uss_mb",
    "apos_category_uss_mb", "apos_del_gc_uss_mb",
]])

grupo | Arrow buffers: 24.3 MB | pandas deep: 24.4 MB | category deep: 12.4 MB | USS após imports: 54.3 MB | USS após Arrow: 203.4 MB | USS após pandas: 205.3 MB | USS após category: 206.5 MB
dsp | Arrow buffers: 26.2 MB | pandas deep: 26.4 MB | category deep: 11.8 MB | USS após imports: 54.4 MB | USS após Arrow: 201.5 MB | USS após pandas: 203.8 MB | USS após category: 205.4 MB


,base,arrow_linhas,arrow_nbytes_mb,arrow_buffers_mb,pandas_deep_mb,pandas_category_deep_mb,tempo_arrow_leitura_s,tempo_arrow_para_pandas_s,tempo_category_s,apos_imports_uss_mb,apos_arrow_uss_mb,apos_pandas_uss_mb,apos_category_uss_mb,apos_del_gc_uss_mb
0,grupo,205400,24.269291,24.269329,24.440816,12.350995,0.047963,0.002951,0.026513,54.339844,203.445312,205.261719,206.546875,204.976562
1,dsp,263761,26.249562,26.249600,26.438343,11.833518,0.053900,0.003066,0.031804,54.390625,201.460938,203.835938,205.429688,203.414062


## 5. Decomposição do aumento de USS

Calcula as diferenças entre as fotografias do processo. A diferença `USS após pandas - USS após Arrow` é uma estimativa do custo incremental da conversão no processo, não uma medida exata de alocação exclusiva da função.

In [5]:
decomp = []
for r in resultados:
    decomp.append({
        "base": r["base"],
        "delta_uss_arrow_mb": r["apos_arrow_uss_mb"] - r["apos_imports_uss_mb"],
        "delta_uss_conversao_pandas_mb": r["apos_pandas_uss_mb"] - r["apos_arrow_uss_mb"],
        "delta_uss_category_mb": r["apos_category_uss_mb"] - r["apos_pandas_uss_mb"],
        "delta_uss_total_apos_imports_mb": r["apos_category_uss_mb"] - r["apos_imports_uss_mb"],
        "uss_retido_apos_del_mb": r["apos_del_gc_uss_mb"] - r["apos_imports_uss_mb"],
        "arrow_buffers_mb": r["arrow_buffers_mb"],
        "pandas_deep_mb": r["pandas_deep_mb"],
        "category_deep_mb": r["pandas_category_deep_mb"],
    })
df_decomp = pd.DataFrame(decomp)
display(df_decomp)

,base,delta_uss_arrow_mb,delta_uss_conversao_pandas_mb,delta_uss_category_mb,delta_uss_total_apos_imports_mb,uss_retido_apos_del_mb,arrow_buffers_mb,pandas_deep_mb,category_deep_mb
0,grupo,149.105469,1.816406,1.285156,152.207031,150.636719,24.269329,24.440816,12.350995
1,dsp,147.070312,2.375000,1.593750,151.039062,149.023438,26.249600,26.438343,11.833518


## 6. Comparação direta com `pd.read_parquet()`

O teste abaixo usa outro processo limpo e mede a leitura pandas direta, para comparar com o caminho Arrow explícito. O objetivo é verificar se `pd.read_parquet()` apresenta um perfil de memória semelhante ao caminho `ParquetFile → Table → to_pandas()`.

In [6]:
WORKER_PANDAS = r'''
import sys, json, gc, time
import pandas as pd
import psutil

p = json.loads(sys.argv[1])
proc = psutil.Process()
def mem():
    x = proc.memory_full_info()
    return {"rss": x.rss / 1024**2, "uss": getattr(x, "uss", 0) / 1024**2}

m0 = mem()
t0 = time.perf_counter()
df = pd.read_parquet(
    p["arquivo"], engine="pyarrow",
    columns=p["columns"], filters=p["filters"]
)
t1 = time.perf_counter()
m1 = mem()
deep = df.memory_usage(index=True, deep=True).sum() / 1024**2
del df
gc.collect()
m2 = mem()
print(json.dumps({
    "base": p["base"],
    "linhas": len(pd.read_parquet(p["arquivo"], engine="pyarrow", columns=["ano"], filters=p["filters"])),
    "tempo_s": t1-t0,
    "delta_rss_mb": m1["rss"]-m0["rss"],
    "delta_uss_mb": m1["uss"]-m0["uss"],
    "deep_mb": deep,
    "uss_retido_mb": m2["uss"]-m0["uss"],
}))
'''

def executar_pandas_direto(base, referencia=REFERENCIA):
    f = filtros_ref(referencia)
    payload = {
        "base": base,
        "arquivo": str(ARQUIVOS[base]),
        "columns": COLUNAS[base],
        "filters": f,
    }
    proc = subprocess.run(
        [sys.executable, "-c", WORKER_PANDAS, json.dumps(payload)],
        cwd=str(PROJECT_ROOT), capture_output=True, text=True
    )
    if proc.returncode:
        raise RuntimeError(proc.stderr)
    linhas = [x for x in proc.stdout.splitlines() if x.strip().startswith("{")]
    return json.loads(linhas[-1])

resultados_direto = [executar_pandas_direto(b) for b in ["grupo", "dsp"]]
display(pd.DataFrame(resultados_direto))

,base,linhas,tempo_s,delta_rss_mb,delta_uss_mb,deep_mb,uss_retido_mb
0,grupo,205400,0.051581,180.140625,166.101562,24.440821,164.550781
1,dsp,263761,0.055791,180.445312,166.597656,26.438348,164.597656


## 7. Arrow dictionary encoding para colunas textuais

Este teste verifica uma alternativa anterior à conversão pandas: aplicar `pyarrow.compute.dictionary_encode()` às colunas textuais com baixa cardinalidade e converter a tabela resultante para pandas.

É um teste experimental de representação, não uma alteração recomendada no pipeline. A codificação dictionary pode reduzir o tamanho de colunas repetitivas, mas também tem custo de CPU e pode não ser vantajosa em colunas de alta cardinalidade.

In [7]:
WORKER_DICT = r'''
import sys, json, gc, time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import psutil

p = json.loads(sys.argv[1])
proc = psutil.Process()
def mem():
    x = proc.memory_full_info()
    return {"rss": x.rss / 1024**2, "uss": getattr(x, "uss", 0) / 1024**2}

m0 = mem()
table = pq.ParquetFile(p["arquivo"]).read_row_group(0, columns=p["columns"])
if p["referencia"] == "ESC":
    mask = pc.is_in(table["TIPO_REFERENCIA_TERRITORIAL"],
                    value_set=pa.array(["ESC", "RESIDENCIA"]))
    table = table.filter(mask)
elif p["referencia"] == "PROVA":
    mask = pc.is_in(table["TIPO_REFERENCIA_TERRITORIAL"],
                    value_set=pa.array(["PROVA"]))
    table = table.filter(mask)
m1 = mem()

dict_cols = []
for field in table.schema:
    name = field.name
    if pa.types.is_string(field.type) or pa.types.is_large_string(field.type):
        arr = table[name].combine_chunks()
        n = pc.count_distinct(arr).as_py()
        ratio = n / max(table.num_rows, 1)
        if ratio < p["limiar"]:
            encoded = pc.dictionary_encode(arr)
            table = table.set_column(
                table.schema.get_field_index(name),
                name,
                pa.chunked_array([encoded])
            )
            dict_cols.append(name)

m2 = mem()
t0 = time.perf_counter()
df = table.to_pandas()
t1 = time.perf_counter()
m3 = mem()

out = {
    "base": p["base"],
    "linhas": table.num_rows,
    "colunas_dictionary": dict_cols,
    "arrow_nbytes_mb": table.nbytes / 1024**2,
    "arrow_buffers_mb": table.get_total_buffer_size() / 1024**2,
    "delta_uss_leitura_arrow_mb": m1["uss"]-m0["uss"],
    "delta_uss_dictionary_mb": m2["uss"]-m1["uss"],
    "delta_uss_to_pandas_mb": m3["uss"]-m2["uss"],
    "pandas_deep_mb": df.memory_usage(index=True, deep=True).sum()/1024**2,
    "tempo_to_pandas_s": t1-t0,
    "dtypes": {str(k):str(v) for k,v in df.dtypes.items()},
}
del df, table
gc.collect()
m4 = mem()
out["uss_retido_apos_del_mb"] = m4["uss"]-m0["uss"]
print(json.dumps(out))
'''

def executar_dictionary(base, referencia=REFERENCIA, limiar=0.5):
    payload = {
        "base": base,
        "arquivo": str(ARQUIVOS[base]),
        "columns": COLUNAS[base],
        "referencia": referencia,
        "limiar": limiar,
    }
    proc = subprocess.run(
        [sys.executable, "-c", WORKER_DICT, json.dumps(payload)],
        cwd=str(PROJECT_ROOT), capture_output=True, text=True
    )
    if proc.returncode:
        raise RuntimeError(f"{proc.stdout}\\n{proc.stderr}")
    linhas = [x for x in proc.stdout.splitlines() if x.strip().startswith("{")]
    return json.loads(linhas[-1])

resultados_dictionary = [executar_dictionary(b) for b in ["grupo", "dsp"]]
display(pd.DataFrame(resultados_dictionary))

,base,linhas,colunas_dictionary,arrow_nbytes_mb,arrow_buffers_mb,delta_uss_leitura_arrow_mb,delta_uss_dictionary_mb,delta_uss_to_pandas_mb,pandas_deep_mb,tempo_to_pandas_s,dtypes,uss_retido_apos_del_mb
0,grupo,205400,"[CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERR...",14.946255,14.946293,147.101562,0.234375,3.238281,12.367983,0.005200,"{'ano': 'Int64', 'CO_MUNICIPIO_TERRITORIO': 'c...",149.019531
1,dsp,263761,"[CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERR...",15.197677,15.197715,145.062500,0.230469,4.128906,11.849904,0.005631,"{'ano': 'Int64', 'CO_MUNICIPIO_TERRITORIO': 'c...",147.421875


## 8. Sensibilidade ao limiar de cardinalidade

Testa diferentes limiares para selecionar colunas textuais que podem ser convertidas para `category`. Cada cenário é executado em processo novo para evitar contaminação entre tentativas.

In [8]:
sensibilidade = []
for base in ["grupo", "dsp"]:
    for limiar in [0.05, 0.10, 0.25, 0.50, 0.75, 1.00]:
        r = executar_worker(base, limiar=limiar)
        sensibilidade.append({
            "base": base,
            "limiar": limiar,
            "colunas_category": ", ".join(r["colunas_category"]),
            "pandas_deep_mb": r["pandas_deep_mb"],
            "category_deep_mb": r["pandas_category_deep_mb"],
            "economia_deep_mb": r["pandas_deep_mb"] - r["pandas_category_deep_mb"],
            "delta_uss_category_mb": r["apos_category_uss_mb"] - r["apos_pandas_uss_mb"],
        })

df_sensibilidade = pd.DataFrame(sensibilidade)
display(df_sensibilidade)

,base,limiar,colunas_category,pandas_deep_mb,category_deep_mb,economia_deep_mb,delta_uss_category_mb
0,grupo,0.05,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI...",24.440816,12.350995,12.089821,1.308594
1,grupo,0.10,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI...",24.440816,12.350995,12.089821,1.304688
2,grupo,0.25,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI...",24.440816,12.350995,12.089821,1.277344
3,grupo,0.50,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI...",24.440816,12.350995,12.089821,1.257812
4,grupo,0.75,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI...",24.440816,12.350995,12.089821,1.214844
5,grupo,1.00,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI...",24.440816,12.350995,12.089821,1.257812
6,dsp,0.05,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI...",26.438343,11.833518,14.604825,1.574219
7,dsp,0.10,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI...",26.438343,11.833518,14.604825,1.476562
8,dsp,0.25,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI...",26.438343,11.833518,14.604825,1.593750
9,dsp,0.50,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI...",26.438343,11.833518,14.604825,1.597656


## 9. Verificação de tipos e cardinalidade por coluna

Esta tabela permite distinguir strings de baixa cardinalidade (por exemplo, sexo, disciplina, tipo de referência) de códigos ou categorias mais diversas. Ela é útil para não converter indiscriminadamente campos como códigos municipais.

In [9]:
def cardinalidade_base(base):
    df = pd.read_parquet(
        ARQUIVOS[base], engine="pyarrow",
        columns=COLUNAS[base], filters=filtros_ref(REFERENCIA)
    )
    rows = []
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or isinstance(df[col].dtype, pd.StringDtype):
            rows.append({
                "base": base,
                "coluna": col,
                "dtype": str(df[col].dtype),
                "n_linhas": len(df),
                "n_unicos": df[col].nunique(dropna=False),
                "razao_unicos": df[col].nunique(dropna=False) / max(len(df), 1),
                "memoria_deep_mb": df[col].memory_usage(index=False, deep=True) / 1024**2,
            })
    del df
    return pd.DataFrame(rows)

df_cardinalidade = pd.concat(
    [cardinalidade_base(b) for b in ["grupo", "dsp"]],
    ignore_index=True
)
display(df_cardinalidade.sort_values(["base", "memoria_deep_mb"], ascending=[True, False]))

,base,coluna,dtype,n_linhas,n_unicos,razao_unicos,memoria_deep_mb
7,dsp,sexo,str,263761,2,0.000008,4.173111
5,dsp,CO_MUNICIPIO_TERRITORIO,string,263761,737,0.002794,3.784028
9,dsp,disciplina,str,263761,6,0.000023,3.002152
6,dsp,TIPO_REFERENCIA_TERRITORIAL,str,263761,1,0.000004,2.798408
8,dsp,faixa_etaria,str,263761,20,0.000076,2.367297
2,grupo,tipo_grupo,str,205400,3,0.000015,3.527943
0,grupo,CO_MUNICIPIO_TERRITORIO,string,205400,673,0.003277,2.950868
4,grupo,disciplina,str,205400,6,0.000029,2.346464
3,grupo,grupo,str,205400,27,0.000131,2.257249
1,grupo,TIPO_REFERENCIA_TERRITORIAL,str,205400,2,0.000010,2.192743


## 10. Resumo e critérios de interpretação

### Como interpretar os resultados

- **USS aumenta já na leitura Arrow:** investigar buffers, descompressão, alocação do leitor e tamanho das colunas físicas.
- **Arrow aumenta pouco, mas USS sobe na conversão para pandas:** o custo dominante está na materialização dos arrays/strings do pandas.
- **`DataFrame deep` é muito menor que o delta USS:** há alocações temporárias, overhead do runtime ou buffers/objetos não representados integralmente pela métrica profunda.
- **Category reduz `deep`, mas não reduz USS:** a representação final fica menor, mas o pico de conversão/cópia pode continuar alto.
- **Dictionary Arrow reduz buffers e USS:** pode justificar um teste de integração posterior, desde que a semântica e os tipos resultantes sejam compatíveis com o dashboard.
- **USS continua alto após `del`:** comparar com a memória inicial do processo e verificar retenção do alocador; não concluir automaticamente que há vazamento.

### Resultado que precisamos obter

Para cada uma das bases `grupo` e `dsp`, identificar qual etapa responde pela maior parcela do aumento de USS:

1. Parquet → Arrow;
2. Arrow → pandas;
3. conversão para category.

Não faça alterações no `features_names.py` ainda. Primeiro vamos interpretar as medições.